In [1]:
import uuid
import logging
from base64 import b64encode
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO)
logging.getLogger(__name__).setLevel(logging.DEBUG)
logging.getLogger("httpx").setLevel(logging.WARNING)
logger = logging.getLogger(__name__)
load_dotenv()

True

## Evaluation pipeline

### PREPARE DATA

In [2]:
from evals import Dataset

dataset_name = "test"
ds_custom = Dataset(dataset_name)
ds_baseline = Dataset(dataset_name)

data_custom = ds_custom.load_dataset()
data_baseline = ds_baseline.load_dataset()

if not data_custom or "sessions" not in data_custom:
    print("Ingen sessions funnet")
    exit()

ds_custom.assign_session_attachments()
ds_baseline.assign_session_attachments_baseline()

total_attachments = sum(len(s.get("attachments", [])) for s in data_custom["sessions"])
logger.info(f"Done — {len(data_custom['sessions'])} sessions, {total_attachments} attachments assigned in total")

INFO:pikepdf._core:pikepdf C++ to Python logger bridge initialized
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "output_schema" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "stream" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
INFO:evals.dataset:Found 18 files under datasets/test/01_data/
INFO:evals.dataset:Session Prosjekt-initialisering | – → 2020-03-15 | 18 candidates, 18 new
INFO:evals.dataset:Found 18 files under datasets/test/01_data/
INFO:evals.dataset:Session Prosjekt-initialisering | – → 2020-03-15 | 18 attachments (cumulative)
INFO:__main__:Done —

In [3]:
ds_custom.data

{'project_id': 'b3f45644-6222-4593-8955-cf4d8e0b00d5',
 'dataset_name': 'test',
 'user_id': '53d63d18-cfa1-416e-96e8-770c8f66507b',
 'last_updated': '2026-02-25T12:55:16.732547+01:00',
 'sessions': [{'session': 0,
   'date': '2020-03-15',
   'session_id': '3eb54a6b-ca3d-4d21-9ea1-b48f4e0228e1',
   'session_name': 'Prosjekt-initialisering',
   'init_query': 'Jeg er advokat og representerer kjøperparet Anders og Berit Kristiansen i en eiendomskjøpssak. De kjøpte en eiendom på Fjellveien 42A i Stavanger kommune den 1. juni 2019, med overtakelse 1. august 2019. \nVi er nå i mars 2020 og det har dukket opp flere problemer med eiendommen.  Jeg trenger din hjelp til å organisere saksinnholdet, identifisere de juridiske problemstillingene, og vurdere mulige tiltak.',
   'conversation': [{'input': 'Gi meg en kort og konsis oppsummering av sakens faktiske bakgrunn og utvikling så langt, basert på dokumentene jeg har lastet opp. \nFokuser på de viktigste hendelsene og problemstillingene. Hva er k

In [4]:
ds_baseline.data

{'project_id': 'b3f45644-6222-4593-8955-cf4d8e0b00d5',
 'dataset_name': 'test',
 'user_id': '53d63d18-cfa1-416e-96e8-770c8f66507b',
 'last_updated': '2026-02-25T12:55:16.732547+01:00',
 'sessions': [{'session': 0,
   'date': '2020-03-15',
   'session_id': '3eb54a6b-ca3d-4d21-9ea1-b48f4e0228e1',
   'session_name': 'Prosjekt-initialisering',
   'init_query': 'Jeg er advokat og representerer kjøperparet Anders og Berit Kristiansen i en eiendomskjøpssak. De kjøpte en eiendom på Fjellveien 42A i Stavanger kommune den 1. juni 2019, med overtakelse 1. august 2019. \nVi er nå i mars 2020 og det har dukket opp flere problemer med eiendommen.  Jeg trenger din hjelp til å organisere saksinnholdet, identifisere de juridiske problemstillingene, og vurdere mulige tiltak.',
   'conversation': [{'input': 'Gi meg en kort og konsis oppsummering av sakens faktiske bakgrunn og utvikling så langt, basert på dokumentene jeg har lastet opp. \nFokuser på de viktigste hendelsene og problemstillingene. Hva er k

### GATHER RESULTS

In [5]:
from evals import CollectAgentResult

In [6]:
car_custom = CollectAgentResult(data_custom, llm_model="google_gemini-2.5-flash", custom_agent=True)
await car_custom.run_agent(embed_to_vectorstore=False, save_to_storage=False)

car_baseline = CollectAgentResult(data_baseline, llm_model="google_gemini-2.5-flash", custom_agent=False)
await car_baseline.run_agent(embed_to_vectorstore=False, save_to_storage=False)

INFO:evals.dataset:Agent initialized with AsyncPostgresSaver checkpointer
INFO:evals.dataset:=========== STARTING EVALUATION ===========
INFO:evals.dataset:Dataset: test | LLM Model: google_gemini-2.5-flash | Custom Agent: True | Sessions: 1 | Project: b3f45644-6222-4593-8955-cf4d8e0b00d5 | User: 53d63d18-cfa1-416e-96e8-770c8f66507b


INFO:evals.dataset:Session 0 | 2020-03-15 | Prosjekt-initialisering | 18 attachments
INFO:agent.context_manager:==== ATTACHMENT ELEMENT DEBUG == 
{'description': 'Damage report from Takst & Analyse AS concerning leakage via the concrete slab forming the roof of the rental unit and the terrace of the main unit. Identifies visible moisture damage, water ingress, accumulated water, and cracks in the slab. The cause is attributed to pillar shoes puncturing the membrane under the terrace tiles. Recommends further investigation and repair.', 'significance': 'high', 'party_roles': ['Anders Kristiansen', 'Berit Kristiansen', 'Takst & Analyse AS'], 'deadlines': []

Storage endpoint URL should have a trailing slash.


INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63d18-cfa1-416e-96e8-770c8f66507b/16ad7793-b40b-4b59-b112-e2cb5925384f/4e8b7886-c9e5-4c3f-9913-a9f6fdee2597.txt
INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63d18-cfa1-416e-96e8-770c8f66507b/16ad7793-b40b-4b59-b112-e2cb5925384f/ccb5ce62-5eae-4894-9ab4-c76591b7c69c.txt
INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63d18-cfa1-416e-96e8-770c8f66507b/16ad7793-b40b-4b59-b112-e2cb5925384f/98ff16f4-3d37-4a3d-bb79-6378543d1855.txt
INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63d18-cfa1-416e-96e8-770c8f66507b/16ad7793-b40b-4b59-b112-e2cb5925384f/fda42c32-3956-444f-9151-76917b9bdbac.txt
INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63d18-cfa1-416e-96e8-770c8f66507b/16ad7793-b40b-4b59-b112-e2cb5925384f/0c158adc-2f57-41ca-81c9-6ee64775b07f.txt
INFO:database.storage_modules:Attachment saved to Supabase Storage at 53d63

In [7]:
from langsmith import Client as LangSmithClient
import os

def get_token_counts(eval_run_id: str) -> dict:
    """Query LangSmith for total token usage across an eval run."""
    client = LangSmithClient()
    project_name = os.getenv("LANGCHAIN_PROJECT", "default")
    runs = list(client.list_runs(
        project_name=project_name,
        filter=f'has(tags, "{eval_run_id}")',
        run_type="llm",
    ))
    return {
        "eval_run_id": eval_run_id,
        "input_tokens": sum(r.prompt_tokens or 0 for r in runs),
        "output_tokens": sum(r.completion_tokens or 0 for r in runs),
        "total_tokens": sum(r.total_tokens or 0 for r in runs),
        "llm_calls": len(runs),
    }

token_counts_custom = get_token_counts(car_custom.data["eval_run_id"])
car_custom.data["token_counts"] = token_counts_custom   
ds_custom.save_results(car_custom.data)
print(token_counts_custom)

INFO:evals.dataset:Results saved to datasets/test/04_results/google_gemini-2.5-flash_custom_2026-02-25_14-26-45.json


{'eval_run_id': '508f6d94-2515-47ec-95d0-f7bba8eaaad0', 'input_tokens': 285396, 'output_tokens': 33877, 'total_tokens': 319273, 'llm_calls': 16}


In [8]:
token_counts_baseline = get_token_counts(car_baseline.data["eval_run_id"])
car_baseline.data["token_counts"] = token_counts_baseline
ds_custom.save_results(car_baseline.data)
print(token_counts_baseline)

INFO:evals.dataset:Results saved to datasets/test/04_results/google_gemini-2.5-flash_baseline_2026-02-25_14-27-06.json


{'eval_run_id': 'bd211f50-2b7c-42fe-a4c7-bcf95a3f52eb', 'input_tokens': 69076, 'output_tokens': 22885, 'total_tokens': 91961, 'llm_calls': 7}


### EVALUATE RESULTS

In [13]:
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval
from deepeval import evaluate
from deepeval.test_case import ConversationalTestCase, Turn, TurnParams
from deepeval.metrics import ConversationalGEval

In [47]:
from evals import Dataset
dataset_name = "test"
ds = Dataset(dataset_name)
collected_results = ds.load_results()
result_file = list(collected_results.keys())
result_file

['datasets/test/04_results/google_gemini-2.5-flash_baseline_2026-02-25_14-27-06.json',
 'datasets/test/04_results/google_gemini-2.5-flash_custom_2026-02-25_14-26-45.json']

In [ ]:

collected_result = collected_results.get(result_file[0]) if result_file else None
session1 = collected_result["sessions"][0]
conv1 = session1["conversation"][0]
conv1

{'input': 'Gi meg en kort og konsis oppsummering av sakens faktiske bakgrunn og utvikling så langt, basert på dokumentene jeg har lastet opp. \nFokuser på de viktigste hendelsene og problemstillingene. Hva er kjernen i saken?',
 'answer': "Anders og Berit Kristiansen kjøpte en eiendom på Fjellveien 42A i Stavanger kommune den 1. juni 2019, med overtakelse 1. august 2019.\n\nSelger meldte fra om vannlekkasje (juli 2019), altså før overtagelse, hvor lekkasjen skal ha blitt forsøkt utbedret i forkant av overtagelsen. \nKun 18 dager etter overtagelsen var lekkasjen tilbake og dette ble meldt fra om av kjøper til selgeren pr epost (18.aug)\nDet ble utarbeidet en skaderapporter (10.sept 2019) som tar for seg vannlekkasjen og de omkringliggende bygningsdelene. Rapporten konkluderer med at membranen var punktert av søyleskoene og at det er årsaken til vanngjennomtrengningen.\n\nSiste oppdatering er fra mars 2020 hvor det ble laget en rapport på betongdekket over utleiedelen for å avklare om de

In [26]:
from deepeval.evaluate.types import EvaluationResult
from google.cloud import storage
from deepeval.evaluate import AsyncConfig
import json

In [58]:

class Evaluater:
    def __init__(self, client=None, bucket_name="master-thesis-prod"):
        self._client = client or storage.Client()
        self.bucket = self._client.bucket(bucket_name)

    def collect_single(self, conversation_turn: dict, session_name: str = "unknown") -> LLMTestCase | None:
        if "input" not in conversation_turn or "model_response" not in conversation_turn or "answer" not in conversation_turn:
            raise ValueError("Conversation turn must contain 'input', 'model_response', and 'answer' fields.")
        if not conversation_turn.get("input") or not conversation_turn.get("model_response") or not conversation_turn.get("answer"):
            logger.warning("Conversation turn is missing required fields. Skipping evaluation for this turn.")
            return None
        return LLMTestCase(
            name=f"Turn {conversation_turn.get('order', 'unknown')} in session {session_name}",
            input=conversation_turn.get("input"),
            actual_output=conversation_turn.get("model_response"),
            expected_output=conversation_turn.get("answer"),
            additional_metadata={
                "turn_order": conversation_turn.get("order", "unknown"),
                "query_id": conversation_turn.get("query_id", "unknown"),
            },
        )

    def eval_conversation(self, conversation: list[dict]):
        turns = []
        for item in conversation:
            turns.append(Turn(role="user", content=item["input"]))
            turns.append(Turn(role="assistant", content=item["model_response"]))

        convo_test_case = ConversationalTestCase(turns=turns)

        metric = ConversationalGEval(
            name="Legal Accuracy",
            criteria="Evaluate whether the assistant's legal analysis is accurate and consistent across the conversation.",
            evaluation_params=[TurnParams.CONTENT],
            threshold=0.5,
        )

        return evaluate(test_cases=[convo_test_case], metrics=[metric], async_config=AsyncConfig(max_concurrent=2, throttle_value=3))

    def run_session_eval(self, session: dict):
        test_cases = [
            tc for conv in session.get("conversation", [])
            if (tc := self.collect_single(conversation_turn=conv, session_name=session.get("session_name", "unknown"))) is not None
        ]

        correctness = GEval(
            name="correctness",
            criteria="Determine if actual_output is factually correct based on expected_output",
            evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
            threshold=0.5,
        )

        return evaluate(test_cases=test_cases, metrics=[correctness], async_config=AsyncConfig(max_concurrent=2, throttle_value=3))

    def run_evaluation(self,data : dict):
        sessions = data.get("sessions", [])
        results = []
        for session in sessions:
            results.append(self.run_session_eval(session))
        return results

    def save_evaluation_results(self, results : list[EvaluationResult], data : dict):
        output = {
            "dataset_name": data.get("dataset_name"),
            "project_id": data.get("project_id"),
            "user_id": data.get("user_id"),
            "eval_run_id": data.get("eval_run_id"),
            "llm_model": data.get("llm_model"),
            "custom_agent": data.get("custom_agent"),
            "token_counts" : data.get("token_counts"),
            "results": [r.model_dump() for r in results if isinstance(r, EvaluationResult)] if results else None,
            }
        agent_type = "custom" if data.get("custom_agent") else "baseline"
        filepath = f'datasets/{data.get("dataset_name")}/05_evals/{data.get("llm_model")}_{agent_type}_{data.get("eval_run_id")}.json'
        blob = self.bucket.blob(filepath)
        blob.upload_from_string(json.dumps(output, indent=4, ensure_ascii=False), content_type='application/json')
        return output


In [59]:
e = Evaluater()

In [51]:
result = e.run_evaluation(data=collected_result)

✨ You're running DeepEval's latest correctness [GEval] Metric! (using gpt-4.1, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases


INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases


INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases


INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases


INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases




Metrics Summary

  - ✅ correctness [GEval] (score: 0.9970687768718666, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The actual output aligns closely with the expected output, accurately summarizing the key facts: property address, purchase and takeover dates, seller's notification of a leak before takeover, attempted repairs, recurrence of the leak, subsequent damage reports, and the findings of structural deficiencies in the concrete deck. It also correctly identifies the core issue as hidden structural and water damage problems not disclosed at sale. No factual inaccuracies or omissions are present., error: None)

For test case:

  - input: Gi meg en kort og konsis oppsummering av sakens faktiske bakgrunn og utvikling så langt, basert på dokumentene jeg har lastet opp. 
Fokuser på de viktigste hendelsene og problemstillingene. Hva er kjernen i saken?
  - actual output: Basert på de opplastede dokumentene er sakens kjerne følgende:

**Faktisk bakgrunn og utvikli

⚠ WARNING: No hyperparameters logged.
» ]8;id=509110;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=196508;https://app.confident-ai.com/project/cmlt7bho400hsmx1e0kg8cu1y/test-runs/cmm24707o005pqg1ejwgz7ocy/regression-testing\https://app.confident-ai.com/project/cmlt7bho400hsmx1e0kg8cu1y/test-runs/cmm24707o005pqg1ejwgz7ocy/regression-testi]8;;\
]8;id=196508;https://app.confident-ai.com/project/cmlt7bho400hsmx1e0kg8cu1y/test-runs/cmm24707o005pqg1ejwgz7ocy/regression-testing\ng]8;;\

In [54]:
result

[EvaluationResult(test_results=[TestResult(name='Turn 1 in session Prosjekt-initialisering', success=True, metrics_data=[MetricData(name='correctness [GEval]', threshold=0.5, success=True, score=0.9970687768718666, reason="The actual output aligns closely with the expected output, accurately summarizing the key facts: property address, purchase and takeover dates, seller's notification of a leak before takeover, attempted repairs, recurrence of the leak, subsequent damage reports, and the findings of structural deficiencies in the concrete deck. It also correctly identifies the core issue as hidden structural and water damage problems not disclosed at sale. No factual inaccuracies or omissions are present.", strict_mode=False, evaluation_model='gpt-4.1', error=None, evaluation_cost=0.00475, verbose_logs='Criteria:\nDetermine if actual_output is factually correct based on expected_output \n \nEvaluation Steps:\n[\n    "Review the expected_output to identify the correct factual informati

In [60]:
e.save_evaluation_results(result, collected_result)

{'dataset_name': 'test',
 'project_id': 'b3f45644-6222-4593-8955-cf4d8e0b00d5',
 'user_id': '53d63d18-cfa1-416e-96e8-770c8f66507b',
 'eval_run_id': 'bd211f50-2b7c-42fe-a4c7-bcf95a3f52eb',
 'llm_model': 'google_gemini-2.5-flash',
 'custom_agent': False,
 'token_counts': {'eval_run_id': 'bd211f50-2b7c-42fe-a4c7-bcf95a3f52eb',
  'input_tokens': 69076,
  'output_tokens': 22885,
  'total_tokens': 91961,
  'llm_calls': 7},
 'results': [{'test_results': [{'name': 'Turn 1 in session Prosjekt-initialisering',
     'success': True,
     'metrics_data': [{'name': 'correctness [GEval]',
       'threshold': 0.5,
       'success': True,
       'score': 0.9970687768718666,
       'reason': "The actual output aligns closely with the expected output, accurately summarizing the key facts: property address, purchase and takeover dates, seller's notification of a leak before takeover, attempted repairs, recurrence of the leak, subsequent damage reports, and the findings of structural deficiencies in the c